<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/03-oop-for-ml/04_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: Build a Tiny Autograd Engine and Train a Neural Network

In PyTorch, training comes down to three lines:

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

`loss.backward()` computes the gradient of the loss with respect to every weight in the model, even a model with billions of weights, automatically. It feels like magic, but the idea behind it is simple enough to build in an afternoon with what you learned in this module.

In this project you'll write:

1. a `Value` class that records every calculation it's part of, using operator overloading (`__add__`, `__mul__` and friends)
2. a `backward()` method that walks back through those calculations and computes gradients using the chain rule
3. `Neuron`, `Layer` and `MLP` classes, organised the way PyTorch's `nn.Module` is
4. an `SGD` optimizer class
5. a training loop that teaches the network to classify points inside and outside a circle

Everything is pure Python, about 150 lines in total. The only maths you need is the derivative of a few simple functions, and they're all given.

## Step 1: What is a gradient, and why do we need one?

Training a model means adjusting its weights to make the loss smaller. To do that, for each weight we need to know: if I nudge this weight up a little, does the loss go up or down, and by how much? That number is the **gradient** (or derivative) of the loss with respect to that weight.

You can estimate it directly: nudge the input by a tiny amount `h` and see how much the output changes.

In [1]:
def f(x):
    return 3 * x ** 2 - 4 * x + 5

def numerical_gradient(f, x, h=1e-6):
    return (f(x + h) - f(x)) / h

print(numerical_gradient(f, 2.0))   # the exact answer is 6x - 4 = 8

8.000003001384925


That works for one input. But a neural network has millions of weights, and nudging each one separately would mean running the whole network millions of times per training step. Far too slow.

**Backpropagation** gets all the gradients at once, in a single backwards pass through the calculation. The key is the **chain rule**: if `c` depends on `b`, and `b` depends on `a`, then

$$\frac{dc}{da} = \frac{dc}{db} \cdot \frac{db}{da}$$

In words: to find out how `a` affects the final result, multiply how `b` affects the result by how `a` affects `b`. Every operation only needs to know its own small, local derivative. The chain rule stitches them together.

So the plan is:

1. As we compute the loss, **record** every operation: what its inputs were and how to compute its local derivative.
2. Then walk **backwards** from the loss, multiplying local derivatives along the way, until every input has its gradient.

Recording every operation is exactly what operator overloading lets us do. When we write `a * b`, our `__mul__` method runs, so it can remember that `a` and `b` were multiplied.

## Step 2: The Value class

A `Value` wraps a single number. It stores:

- `data`: the number itself
- `grad`: the gradient of the final output with respect to this value, filled in by `backward()`
- `_parents`: the values it was computed from
- `_backward`: a function that knows how to pass gradient back to the parents

Let's start with addition and multiplication.

In [2]:
import math
import random


class Value:
    def __init__(self, data, parents=(), op=""):
        self.data = float(data)
        self.grad = 0.0
        self._parents = parents
        self._op = op
        self._backward = lambda: None     # leaf values have nothing to pass back

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            # d(a + b)/da = 1 and d(a + b)/db = 1, so both just receive out.grad
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            # d(a * b)/da = b and d(a * b)/db = a
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

There's a lot packed into this, so let's go through it.

**`other = other if isinstance(other, Value) else Value(other)`** lets us write `a * 2` as well as `a * b`. A plain number gets wrapped in a `Value`.

**`out = Value(..., (self, other), "*")`** creates the result and records where it came from. Every result knows its parents, so together they form a graph of the whole calculation.

**The inner `_backward` function** is defined inside `__mul__`, so it can see the variables `self`, `other` and `out` from that moment. A function that remembers the variables from where it was created is called a *closure*. We store it on `out`, and when we call it later, it still knows exactly which two values were multiplied.

**Inside `_backward`**, each parent gets `local derivative × out.grad`. That's the chain rule. `out.grad` is how much the final result changes when `out` changes, and the local derivative is how much `out` changes when this parent changes.

**`+=`, not `=`.** This is important, and we'll come back to it in step 4.

Let's build a small expression by hand:

In [3]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)

d = a * b        # -6
e = d + c        # 4
print(e)
print(e._op, e._parents)

Value(data=4.0000, grad=0.0000)
+ (Value(data=-6.0000, grad=0.0000), Value(data=10.0000, grad=0.0000))


`e` remembers it was made by `+` from `d` and `c`, and `d` remembers it was made by `*` from `a` and `b`. Now let's run the backward pass by hand, starting from the end. The gradient of `e` with respect to itself is 1.

In [4]:
e.grad = 1.0
e._backward()     # passes gradient to d and c
d._backward()     # passes gradient to a and b

for name, v in [("a", a), ("b", b), ("c", c), ("d", d)]:
    print(name, v)

a Value(data=2.0000, grad=-3.0000)
b Value(data=-3.0000, grad=2.0000)
c Value(data=10.0000, grad=1.0000)
d Value(data=-6.0000, grad=1.0000)


Check against the maths: `e = a*b + c`, so `de/da = b = -3`, `de/db = a = 2` and `de/dc = 1`. All correct.

We had to call `_backward` in the right order by hand: `e` before `d`, because `d` can't pass its gradient on until it has received all of its own. For a big network we need to work that order out automatically.

## Step 3: backward() and topological order

The rule is: a value's `_backward` can only run once every value that *uses* it has already run. An ordering of a graph that respects this kind of rule is called a **topological order**.

We can build one with a depth-first search: before adding a value to the list, first add all of its parents. That gives an order where parents always come before their children. Then we go through that list in reverse.

We'll add `backward()` to the class as a function and attach it, so we don't have to redefine everything. In a real project you'd just write it inside the class.

In [5]:
def backward(self):
    order = []
    visited = set()

    def visit(v):
        if v not in visited:
            visited.add(v)
            for parent in v._parents:
                visit(parent)
            order.append(v)

    visit(self)

    self.grad = 1.0
    for v in reversed(order):
        v._backward()


Value.backward = backward

`visited` is a set, so checking whether we've already seen a value is O(1), the lesson from Module 2. A network has tens of thousands of these values per training step, so that matters.

You can add a set of `Value` objects even though `Value` is a class we wrote, because we didn't define `__eq__`. By default, objects are hashable and compare by identity, which is exactly what we want here: two different values that happen to hold the same number are still different nodes in the graph.

Let's try it on the same expression:

In [6]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
e = a * b + c
e.backward()
print(a.grad, b.grad, c.grad)

-3.0 2.0 1.0


Same answer, no manual steps.

## Step 4: Why += matters

What if a value is used more than once? Take `b = a + a`. The derivative of `2a` is 2, so `a.grad` should be 2.

In [7]:
a = Value(3.0)
b = a + a
b.backward()
print(a.grad)

2.0


It's 2, because the `_backward` for that addition runs `self.grad += out.grad` and then `other.grad += out.grad`, and `self` and `other` are both `a`. Each use adds its share.

If we had written `self.grad = out.grad`, the second assignment would overwrite the first, and `a.grad` would be 1. Wrong, and with no error message. In a neural network, every input feeds into many neurons, so every value is used many times. With `=` instead of `+=`, nothing would learn properly.

The flip side of accumulating is that gradients pile up if you call `backward()` twice without resetting them:

In [8]:
a = Value(3.0)
y = a * a          # dy/da = 2a = 6
y.backward()
print(a.grad)
y.backward()
print(a.grad)      # 12: the second pass added on top of the first

6.0
12.0


That's exactly why PyTorch training loops call `optimizer.zero_grad()` before every `loss.backward()`. PyTorch accumulates gradients for the same reason we do, and forgetting `zero_grad()` is a classic bug: the model still trains, just badly, because every step uses the sum of all previous gradients.

## Step 5: More operations

A neural network needs a few more operations: subtraction, division, powers, and an activation function. We'll use **tanh**, which squashes any number into the range -1 to 1.

Most of these can be built from what we already have. `a - b` is `a + (-b)`, and `-b` is `b * -1`. `a / b` is `a * b**-1`. So only the power and tanh need their own derivatives:

- `d(x**k)/dx = k * x**(k-1)`
- `d(tanh(x))/dx = 1 - tanh(x)**2`

We also need the reflected versions (`__radd__`, `__rmul__` and so on) so that expressions like `2 * a` and `1 - a` work when the plain number is on the left. And `__radd__` has one more important job: Python's `sum()` starts from `0`, so `sum(values)` begins with `0 + values[0]`, which only works if `Value` has `__radd__`.

In [9]:
def __pow__(self, k):
    assert isinstance(k, (int, float)), "only number exponents are supported"
    out = Value(self.data ** k, (self,), f"**{k}")

    def _backward():
        self.grad += k * self.data ** (k - 1) * out.grad

    out._backward = _backward
    return out


def tanh(self):
    t = math.tanh(self.data)
    out = Value(t, (self,), "tanh")

    def _backward():
        self.grad += (1 - t ** 2) * out.grad

    out._backward = _backward
    return out


Value.__pow__ = __pow__
Value.tanh = tanh
Value.__neg__ = lambda self: self * -1
Value.__sub__ = lambda self, other: self + (-other)
Value.__truediv__ = lambda self, other: self * other ** -1
Value.__radd__ = lambda self, other: self + other
Value.__rmul__ = lambda self, other: self * other
Value.__rsub__ = lambda self, other: (-self) + other
Value.__rtruediv__ = lambda self, other: Value(other) * self ** -1

For `__truediv__`, `other ** -1` only works if `other` is a `Value`, since a plain number's `**` gives a plain number. That's fine here: `a / 2` becomes `a * 0.5`, and `__mul__` wraps the 0.5.

### Checking the gradients

Whenever you write gradient code, check it against the numerical estimate from step 1. It's the single most useful test for this kind of code. Here's a messy expression that uses every operation:

In [10]:
def expression(x, y):
    return ((x * y + 3) ** 2 / (1 + x * x) - y).tanh() * 2 - x / 4


x, y = Value(0.7), Value(-1.3)
out = expression(x, y)
out.backward()

h = 1e-6
num_dx = (expression(Value(0.7 + h), Value(-1.3)).data - out.data) / h
num_dy = (expression(Value(0.7), Value(-1.3 + h)).data - out.data) / h

print(f"dx: autograd={x.grad:.6f}  numerical={num_dx:.6f}")
print(f"dy: autograd={y.grad:.6f}  numerical={num_dy:.6f}")

dx: autograd=-0.260806  numerical=-0.260806
dy: autograd=0.001627  numerical=0.001627


They match to six decimal places. If you print more digits you'll see tiny differences, and those come from the numerical estimate, which is only approximate. The autograd version is exact.

## Step 6: Neurons, layers and a network

Now we can build a neural network out of `Value`s, organised the way PyTorch does it.

PyTorch has a base class, `nn.Module`, that every layer and model inherits from. Its most important job is `parameters()`, which collects every trainable weight so an optimizer can update them. We'll write a small version:

In [11]:
class Module:
    def parameters(self):
        return []

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

A **neuron** takes several inputs, multiplies each by a weight, adds them up with a bias, and passes the result through tanh:

$$\text{out} = \tanh\left(\sum_i w_i x_i + b\right)$$

Its parameters are its weights and its bias.

In [12]:
class Neuron(Module):
    def __init__(self, n_inputs, rng):
        self.w = [Value(rng.uniform(-1, 1)) for _ in range(n_inputs)]
        self.b = Value(0.0)

    def __call__(self, x):
        total = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return total.tanh()

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f"Neuron({len(self.w)})"

`sum(..., self.b)` uses the optional second argument of `sum`, the starting value. It starts from the bias instead of 0, which saves one addition node per neuron.

A **layer** is a list of neurons that all receive the same inputs. An **MLP** (multi-layer perceptron) is a list of layers, where each layer's outputs are the next layer's inputs. Both are built by composition, and both just collect their children's parameters.

In [13]:
class Layer(Module):
    def __init__(self, n_inputs, n_outputs, rng):
        self.neurons = [Neuron(n_inputs, rng) for _ in range(n_outputs)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer({len(self.neurons[0].w)} -> {len(self.neurons)})"


class MLP(Module):
    def __init__(self, n_inputs, layer_sizes, seed=0):
        rng = random.Random(seed)
        sizes = [n_inputs] + layer_sizes
        self.layers = [Layer(sizes[i], sizes[i + 1], rng) for i in range(len(layer_sizes))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP({', '.join(repr(layer) for layer in self.layers)})"


model = MLP(2, [8, 8, 1])
print(model)
print("parameters:", len(model.parameters()))
print(model([0.5, -1.0]))

MLP(Layer(2 -> 8), Layer(8 -> 8), Layer(8 -> 1))
parameters: 105
Value(data=0.5749, grad=0.0000)


`MLP(2, [8, 8, 1])` means 2 inputs, two hidden layers of 8 neurons, and 1 output. The parameter count is (2 weights + 1 bias) × 8 + (8 + 1) × 8 + (8 + 1) × 1 = 105.

Notice how `parameters()` works through composition: the MLP asks each layer, each layer asks each neuron, and each neuron returns its own weights. That's exactly how `model.parameters()` works in PyTorch, through any depth of nested modules.

### The optimizer

Finally, an optimizer that holds the parameters and updates them. Plain gradient descent moves each parameter a small step *against* its gradient, since the gradient points in the direction that makes the loss bigger:

In [14]:
class SGD:
    def __init__(self, parameters, lr=0.1):
        self.parameters = list(parameters)
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = 0.0

    def step(self):
        for p in self.parameters:
            p.data -= self.lr * p.grad

Note that `step` changes `p.data`, not `p`. The optimizer must update the *same* `Value` objects the model uses. If it created new ones, the model would never see the change. That works because the list the optimizer holds and the lists inside the model contain references to the same objects, the aliasing idea from Module 1, used on purpose this time.

## Step 7: The data

We'll generate points in a square and label them +1 if they're inside a circle and -1 if they're outside. A single straight line can't separate those two groups, so a model needs hidden layers and a non-linear activation to learn it. That's what makes it a good first test for a neural network.

In [15]:
def make_circle_data(n, radius=1.3, seed=0):
    rng = random.Random(seed)
    X, y = [], []
    for _ in range(n):
        point = [rng.uniform(-2, 2), rng.uniform(-2, 2)]
        inside = point[0] ** 2 + point[1] ** 2 < radius ** 2
        X.append(point)
        y.append(1.0 if inside else -1.0)
    return X, y


X_train, y_train = make_circle_data(60, seed=1)
X_test, y_test = make_circle_data(100, seed=2)

print("inside the circle:", sum(v > 0 for v in y_train), "of", len(y_train))

inside the circle: 23 of 60


## Step 8: Train

Here's the training loop. Read it next to a PyTorch training loop and you'll see it's the same shape, line for line:

In [16]:
def accuracy(model, X, y):
    correct = sum((model(x).data > 0) == (target > 0) for x, target in zip(X, y))
    return correct / len(y)


model = MLP(2, [8, 8, 1], seed=0)
optimizer = SGD(model.parameters(), lr=0.1)

for step in range(201):
    # forward pass: mean squared error between outputs and targets
    preds = [model(x) for x in X_train]
    loss = sum((p - t) ** 2 for p, t in zip(preds, y_train)) / len(y_train)

    # backward pass and update
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 25 == 0:
        print(f"step {step:3d}  loss={loss.data:.4f}  train acc={accuracy(model, X_train, y_train):.0%}")

print(f"\ntest accuracy: {accuracy(model, X_test, y_test):.0%}")

step   0  loss=1.5713  train acc=52%


step  25  loss=0.5317  train acc=87%


step  50  loss=0.3578  train acc=92%


step  75  loss=0.2830  train acc=93%


step 100  loss=0.2416  train acc=97%


step 125  loss=0.2138  train acc=98%


step 150  loss=0.1924  train acc=98%


step 175  loss=0.1742  train acc=100%


step 200  loss=0.1579  train acc=100%

test accuracy: 98%


The loss goes down and accuracy goes up. The network has learned the circle from 60 example points, with gradients computed by about a hundred lines of code we wrote ourselves.

It's slow, though. Every single number is a Python object, and each training step creates tens of thousands of them. PyTorch does exactly the same thing conceptually, but each node in its graph is a whole tensor of numbers, and the actual arithmetic runs in optimised C++ or on a GPU. Same idea, millions of times faster.

## Step 9: Look at what it learned

Let's ask the model about every point on a grid and draw its answers with text: `#` where it says "inside" and `.` where it says "outside". The true circle has a radius of 1.3.

In [17]:
def draw(model, size=21, low=-2.0, high=2.0):
    step = (high - low) / (size - 1)
    for row in range(size):
        y = high - row * step
        line = ""
        for col in range(size):
            x = low + col * step
            line += "# " if model([x, y]).data > 0 else ". "
        print(line)


draw(model)

. . . . . . . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . 
. . . . . . . . . # # # . . . . . . . . . 
. . . . . . . # # # # # # # # . . . . . . 
. . . . . . # # # # # # # # # # . . . . . 
. . . . . # # # # # # # # # # # . . . . . 
. . . . . # # # # # # # # # # # # . . . . 
. . . . # # # # # # # # # # # # # . . . . 
. . . . # # # # # # # # # # # # # . . . . 
. . . . # # # # # # # # # # # # # . . . . 
. . . . . # # # # # # # # # # # . . . . . 
. . . . . # # # # # # # # # # # . . . . . 
. . . . . . # # # # # # # # # . . . . . . 
. . . . . . . # # # # # # # . . . . . . . 
. . . . . . . . . # # # # . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . 


A blob of `#` in the middle: the network has worked out that the positive class is a round region in the centre, using nothing but the 60 labelled points. The edge is a bit rough, which is what you'd expect from so little data and such a small network.

## The same thing in PyTorch

For comparison, here's what this project looks like in PyTorch. You don't need to run it. The point is that every piece maps onto something you just built.

```python
import torch
import torch.nn as nn

model = nn.Sequential(                # our MLP
    nn.Linear(2, 8), nn.Tanh(),       # our Layer + Neuron
    nn.Linear(8, 8), nn.Tanh(),
    nn.Linear(8, 1), nn.Tanh(),
)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)   # our SGD

X = torch.tensor(X_train)
y = torch.tensor(y_train).unsqueeze(1)

for step in range(201):
    loss = ((model(X) - y) ** 2).mean()   # builds a graph, like our Value
    optimizer.zero_grad()                 # our zero_grad
    loss.backward()                       # our backward()
    optimizer.step()                      # our step()
```

| PyTorch | This project |
|---|---|
| `torch.Tensor` with `requires_grad` | `Value` |
| autograd graph | `_parents` and `_backward` |
| `loss.backward()` | topological sort, then each `_backward` in reverse |
| `nn.Module`, `parameters()` | `Module`, `parameters()` |
| `nn.Linear` + activation | `Layer` of `Neuron`s |
| `optimizer.zero_grad()` / `step()` | `SGD.zero_grad()` / `step()` |

## Your turn

1. **Add ReLU.** Write a `relu` method: the output is `max(0, x)`, and the local derivative is 1 if the output is positive and 0 otherwise. Use it in the hidden layers instead of tanh (keep tanh on the output). Does the network train faster?

2. **Add `exp` and `log`,** then use them to write a proper classification loss instead of mean squared error. Hint: the derivative of `exp(x)` is `exp(x)`, and the derivative of `log(x)` is `1/x`.

3. **Use your learning-rate schedule.** Plug the `WarmupCosineSchedule` from exercise 11 into the loop by setting `optimizer.lr = schedule(step)` before each update. Does it end at a lower loss?

4. **Momentum.** Write an `SGDMomentum` optimizer that keeps a "velocity" for each parameter: `v = momentum * v + grad`, then `data -= lr * v`. Store the velocities in a dict keyed by parameter. (Why does that work, given that `Value` is a class we wrote? Think about `__hash__` and section 3 of the lesson.)

5. **Mini-batches.** Instead of using all 60 points every step, wrap the data in the `TabularDataset` and `DataLoader` from the lesson and train on batches of 10. Each step is cheaper, so you can run more of them.

6. **Count the graph.** Modify `backward()` to return `len(order)`, the number of nodes in the graph. How many `Value` objects does one training step create? This is the number PyTorch avoids by working with whole tensors at once.

## What you used

- **Operator overloading** (`__add__`, `__mul__`, `__pow__`, `__radd__`, `__neg__`, `__truediv__` and the rest) to record a graph of every calculation
- **Closures** to store each operation's backward rule along with the values it needs
- **`__call__`** to make neurons, layers and the network callable, like PyTorch modules
- **Inheritance**, with `Module` as a shared base class providing `zero_grad`
- **Composition**: a network has layers, a layer has neurons, a neuron has values, and `parameters()` collects them all
- **Sets** for the visited nodes, and **identity-based hashing** for graph nodes
- **Shared references** between the model and the optimizer, used on purpose

## Further reading

The design of this engine follows the same approach as Andrej Karpathy's [micrograd](https://github.com/karpathy/micrograd), a small, very readable autograd library. His video *The spelled-out intro to neural networks and backpropagation: building micrograd* goes through the same ideas in more depth and is well worth watching once you've finished this project.